In [ ]:
import pickle

file_path = './output/traces-f1e3ebbacf16080764549941f282eb36.pkl'

output = []

# Exception handling for loading the pickle file
try:
    with open(file_path, 'rb') as f:
        output = pickle.load(f)
    print(f"File successfully loaded from {file_path}")

except FileNotFoundError as fnf_error:
    print(f"FileNotFoundError: {fnf_error}")
except PermissionError as perm_error:
    print(f"PermissionError: {perm_error}")
except pickle.UnpicklingError as unpickling_error:
    print(f"UnpicklingError: {unpickling_error}")
except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
# AIMessage(content='', 
#           additional_kwargs={'tool_calls': [{'id': 'call_eKoLsmhoONpiJxoeaoNhYt0m', 'function': {'arguments': '{}', 'name': 'transfer_to_sql_agent'}, 'type': 'function'}], 'refusal': None}, 
#           response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 246, 'total_tokens': 259, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-BX2xWRGE9shaR62mXRoHHNRGaB4oW', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}}, 
#           name='supervisor', 
#           id='run--0e7c16bb-8fd2-42d3-9d0c-783806d127e7-0', 
#           tool_calls=[{'name': 'transfer_to_sql_agent', 'args': {}, 'id': 'call_eKoLsmhoONpiJxoeaoNhYt0m', 'type': 'tool_call'}], 
#           usage_metadata={'input_tokens': 246, 'output_tokens': 13, 'total_tokens': 259, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
#           ),
 

In [ ]:
output

In [ ]:
from ragas.integrations.langgraph import convert_to_ragas_messages

ragas_trace = convert_to_ragas_messages(
    output
)
ragas_trace

In [ ]:
from ragas.metrics import ToolCallAccuracy
from ragas.dataset_schema import  MultiTurnSample
from ragas.messages import ToolCall

sample = ragas_trace

sample = MultiTurnSample(
    user_input=sample,
    reference_tool_calls=[
        ToolCall(name="transfer_to_sql_agent", args={}),
        ToolCall(name="check_relevance", args={"question": "Which country's customer spent the most?"}),
        ToolCall(name="convert_nl_to_sql", args={"question": "Which country's customer spent the most?"}),
        ToolCall(name="query_checker", args={"sql_query": "SELECT Customer.Country AS country, SUM(Invoice.Total) AS total_spent\nFROM Customer\nJOIN Invoice ON Customer.CustomerId = Invoice.CustomerId\nGROUP BY Customer.Country\nORDER BY total_spent DESC\nLIMIT 1;"}),
        ToolCall(name="execute_sql", args={"sql_query": "SELECT Customer.Country AS country, SUM(Invoice.Total) AS total_spent\nFROM Customer\nJOIN Invoice ON Customer.CustomerId = Invoice.CustomerId\nGROUP BY Customer.Country\nORDER BY total_spent DESC\nLIMIT 1;"}),
        ToolCall(name="transfer_back_to_supervisor", args={}),
    ]
)

scorer = ToolCallAccuracy()
await scorer.multi_turn_ascore(sample)


In [ ]:
from ragas.metrics import ToolCallAccuracy
from ragas.dataset_schema import  MultiTurnSample
from ragas.messages import ToolCall

sample = ragas_trace

sample = MultiTurnSample(
    user_input=sample,
    reference_tool_calls=[
        ToolCall(name="transfer_to_sql_agent", args={}),
        ToolCall(name="check_relevance", args={"question": "Which country's customer spent the most?"}),
        ToolCall(name="convert_nl_to_sql", args={"question": "Which country's customer spent the most?"}),
        ToolCall(name="query_checker", args={"sql_query": "SELECT c.Country AS country, SUM(i.Total) AS total_spent\nFROM Customer c\nJOIN Invoice i ON c.CustomerId = i.CustomerId\nGROUP BY c.Country\nORDER BY total_spent DESC\nLIMIT 1;"}),
        ToolCall(name="execute_sql", args={"sql_query": "SELECT c.Country AS country, SUM(i.Total) AS total_spent\nFROM Customer c\nJOIN Invoice i ON c.CustomerId = i.CustomerId\nGROUP BY c.Country\nORDER BY total_spent DESC\nLIMIT 1;"}),
        ToolCall(name="transfer_back_to_supervisor", args={}),
    ]
)

scorer = ToolCallAccuracy()
await scorer.multi_turn_ascore(sample)


In [2]:
import os
from dotenv import load_dotenv
from langchain_openai.chat_models import AzureChatOpenAI

load_dotenv()
api_key = os.environ['api_key']
azure_endpoint = os.environ['azure_endpoint']
api_version = os.environ['api_version']

llm = AzureChatOpenAI(
    api_version = api_version,
    azure_endpoint = azure_endpoint,
    azure_deployment = "gpt-4o",
    model_name = "gpt-4o",
    api_key = api_key,
    temperature = 1.0,
)

In [ ]:
from ragas.metrics import TopicAdherenceScore
from ragas.dataset_schema import MultiTurnSample
from ragas.llms import LangchainLLMWrapper

sample = ragas_trace

sample = MultiTurnSample(
    user_input=sample, 
    reference_topics=["Customer"]
)

scorer = TopicAdherenceScore(llm = LangchainLLMWrapper(llm), mode="precision")
await scorer.multi_turn_ascore(sample)


In [ ]:
from ragas.metrics import TopicAdherenceScore
from ragas.dataset_schema import MultiTurnSample
from ragas.llms import LangchainLLMWrapper

sample = ragas_trace

sample = MultiTurnSample(
    user_input=sample, 
    reference_topics=["Customer","Invoice"]
)

scorer = TopicAdherenceScore(llm = LangchainLLMWrapper(llm), mode="precision")
await scorer.multi_turn_ascore(sample)


In [3]:
from ragas.metrics import LLMSQLEquivalence
from ragas.dataset_schema import SingleTurnSample

sample = SingleTurnSample(
    response="""
        SELECT p.product_name, SUM(oi.quantity) AS total_quantity
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        GROUP BY p.product_name;
    """,
    reference="""
        SELECT p.product_name, COUNT(oi.quantity) AS total_quantity
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        GROUP BY p.product_name;
    """,
    reference_contexts=[
        """
        Table order_items:
        - order_item_id: INT
        - order_id: INT
        - product_id: INT
        - quantity: INT
        """,
        """
        Table products:
        - product_id: INT
        - product_name: VARCHAR
        - price: DECIMAL
        """
    ]
)

scorer = LLMSQLEquivalence()
scorer.llm = llm
await scorer.single_turn_ascore(sample)

AttributeError: 'str' object has no attribute 'content'

In [ ]:
from ragas.metrics import AgentGoalAccuracyWithReference
from ragas.dataset_schema import  MultiTurnSample
from ragas.llms import LangchainLLMWrapper

sample = ragas_trace

sample = MultiTurnSample(
    user_input=sample,
    reference="Customers from country spent the most"
)

scorer = AgentGoalAccuracyWithReference(llm = LangchainLLMWrapper(llm))
await scorer.multi_turn_ascore(sample)


In [ ]:
from ragas.metrics import AgentGoalAccuracyWithReference
from ragas.dataset_schema import  MultiTurnSample
from ragas.llms import LangchainLLMWrapper

sample = ragas_trace

sample = MultiTurnSample(
    user_input=sample,
    reference="Customer spent the most in country"
)

scorer = AgentGoalAccuracyWithReference(llm = LangchainLLMWrapper(llm))
await scorer.multi_turn_ascore(sample)


In [ ]:
from ragas.metrics import AgentGoalAccuracyWithoutReference
from ragas.dataset_schema import  MultiTurnSample
from ragas.llms import LangchainLLMWrapper

sample = ragas_trace

sample = MultiTurnSample(
    user_input=sample
)

scorer = AgentGoalAccuracyWithoutReference(llm = LangchainLLMWrapper(llm))
await scorer.multi_turn_ascore(sample)


In [ ]:
output

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from deepeval.test_case import ToolCall

def convert_to_deepeval_messages(output):
    trace={}

    # Extract the first message (HumanMessage) as the input
    input_message = next(step for step in output if isinstance(step, HumanMessage))
    input_text = input_message.content
    trace['input'] = input_text

    # Extract the last message (AIMessage) as the actual output
    output_message = next(step for step in reversed(output) if isinstance(step, AIMessage))
    actual_output = output_message.content
    trace['actual_output'] = actual_output

    # Collect all AI tool calls
    tool_call_map = {}
    for step in output:
        if isinstance(step, AIMessage) and 'tool_calls' in step.additional_kwargs:
            for tool_call in step.additional_kwargs['tool_calls']:
                tool_call_map[tool_call['id']] = {
                    'name': tool_call['function']['name'],
                    'args': tool_call['function']['arguments']
                }

    # Match tool call results (ToolMessages) to tool inputs
    tools_called = []
    for step in output:
        if isinstance(step, ToolMessage):
            tool_call_id = step.tool_call_id
            tool_data = tool_call_map.get(tool_call_id)
            if tool_data:
                args_dict = eval(tool_data['args'])
                # print(args_dict)
                if args_dict == {}:
                    tools_called.append(ToolCall(name=tool_data['name'], input_parameters={"input": ""}, output=step.content))
                else:
                    tools_called.append(ToolCall(name=tool_data['name'], input_parameters=args_dict, output=step.content))

    trace['tools_called'] = tools_called

    return trace

deepeval_trace = convert_to_deepeval_messages(output)
deepeval_trace

In [ ]:
# test_case = LLMTestCase(
#     input="What if these shoes don't fit?",
#     actual_output="We offer a 30-day full refund at no extra cost.",
#     tools_called=[ToolCall(name="WebSearchTool", input_parameters={"query": "shoes size"}), 
#                   ToolCall(name="QueryTool", input_parameters={"query": "shoes fitting"})],
#     expected_tools=[ToolCall(name="WebSearchTool", input_parameters={"query": "shoes size"})]
# )

In [ ]:
from deepeval.metrics import ToolCorrectnessMetric
from deepeval.test_case import LLMTestCase, ToolCallParams, ToolCall

test_case = LLMTestCase(
    input=deepeval_trace['input'],
    actual_output=deepeval_trace['actual_output'],
    tools_called=deepeval_trace['tools_called'],
    expected_tools=[ToolCall(name="transfer_to_sql_agent", input_parameters={"input":""}),
                    ToolCall(name="check_relevance", input_parameters={"question": "Which country's customer spent the most?"}),
                    ToolCall(name="convert_nl_to_sql", input_parameters={"question": "Which country's customer spent the most?"}),
                    ToolCall(name="query_checker", input_parameters={"sql_query": "SELECT Customer.Country AS country, SUM(Invoice.Total) AS total_spent\nFROM Customer\nJOIN Invoice ON Customer.CustomerId = Invoice.CustomerId\nGROUP BY Customer.Country\nORDER BY total_spent DESC\nLIMIT 1;"}),
                    ToolCall(name="execute_sql", input_parameters={"sql_query": "SELECT Customer.Country AS country, SUM(Invoice.Total) AS total_spent\nFROM Customer\nJOIN Invoice ON Customer.CustomerId = Invoice.CustomerId\nGROUP BY Customer.Country\nORDER BY total_spent DESC\nLIMIT 1;"}),
                    ToolCall(name="transfer_back_to_supervisor", input_parameters={"input":""}),
                    ]
)

metric = ToolCorrectnessMetric(
  evaluation_params=[ToolCallParams.INPUT_PARAMETERS],
  should_consider_ordering=True
)

metric.measure(test_case)
print(metric.score)
print(metric.reason)

In [ ]:
from deepeval.metrics import ToolCorrectnessMetric
from deepeval.test_case import LLMTestCase, ToolCallParams, ToolCall

test_case = LLMTestCase(
    input=deepeval_trace['input'],
    actual_output=deepeval_trace['actual_output'],
    tools_called=deepeval_trace['tools_called'],
    expected_tools=[ToolCall(name="transfer_to_sql_agent", input_parameters={"input":""}),
                    ToolCall(name="check_relevance", input_parameters={"question": "Which country's customer spent the most?"}),
                    ToolCall(name="convert_nl_to_sql", input_parameters={"question": "Which country's customer spent the most?"}),
                    ToolCall(name="query_checker", input_parameters={"sql_query": "SELECT Customer.Country AS country, SUM(Invoice.Total) AS total_spent\nFROM Customer\nJOIN Invoice ON Customer.CustomerId = Invoice.CustomerId\nGROUP BY Customer.Country\nORDER BY total_spent DESC\nLIMIT 1;"}),
                    ]
)

metric = ToolCorrectnessMetric(
  evaluation_params=[ToolCallParams.INPUT_PARAMETERS],
  should_consider_ordering=True
)

metric.measure(test_case)
print(metric.score)
print(metric.reason)

In [ ]:
from deepeval.metrics import ToolCorrectnessMetric
from deepeval.test_case import LLMTestCase, ToolCallParams, ToolCall

test_case = LLMTestCase(
    input=deepeval_trace['input'],
    actual_output=deepeval_trace['actual_output'],
    tools_called=deepeval_trace['tools_called'],
    expected_tools=[ToolCall(name="transfer_to_sql_agent", input_parameters={"input":""}),
                    ToolCall(name="check_relevance", input_parameters={"question": "Which country's customer spent the most?"}),
                    ToolCall(name="convert_nl_to_sql", input_parameters={"question": "Which country's customer spent the most?"}),
                    ToolCall(name="execute_sql", input_parameters={"sql_query": "SELECT Customer.Country AS country, SUM(Invoice.Total) AS total_spent\nFROM Customer\nJOIN Invoice ON Customer.CustomerId = Invoice.CustomerId\nGROUP BY Customer.Country\nORDER BY total_spent DESC\nLIMIT 1;"}),
                    ToolCall(name="query_checker", input_parameters={"sql_query": "SELECT Customer.Country AS country, SUM(Invoice.Total) AS total_spent\nFROM Customer\nJOIN Invoice ON Customer.CustomerId = Invoice.CustomerId\nGROUP BY Customer.Country\nORDER BY total_spent DESC\nLIMIT 1;"}),
                    ToolCall(name="transfer_back_to_supervisor", input_parameters={"input":""}),
                    ]
)

metric = ToolCorrectnessMetric(
  evaluation_params=[ToolCallParams.INPUT_PARAMETERS],
  should_consider_ordering=True
)

metric.measure(test_case)
print(metric.score)
print(metric.reason)

In [ ]:
from deepeval.models import DeepEvalBaseLLM
 
class CustomAzureChatOpenAI(DeepEvalBaseLLM):
    def __init__(self, model):
        self.model = model
 
    def load_model(self):
        return self.model
 
    def generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        return chat_model.invoke(prompt).content
 
    async def a_generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        res = await chat_model.ainvoke(prompt)
        return res.content
 
    def get_model_name(self):
        return "Custom Azure OpenAI Model"
 
custom_model = CustomAzureChatOpenAI(llm)

In [ ]:
from deepeval import evaluate
from deepeval.metrics import TaskCompletionMetric
from deepeval.test_case import LLMTestCase

metric = TaskCompletionMetric(
    threshold=0.7,
    model = custom_model,
    include_reason=True
)

# test_case = LLMTestCase(
#     input="Plan a 3-day itinerary for Paris with cultural landmarks and local cuisine.",
#     actual_output=(
#         "Day 1: Eiffel Tower, dinner at Le Jules Verne. "
#         "Day 2: Louvre Museum, lunch at Angelina Paris. "
#         "Day 3: Montmartre, evening at a wine bar."
#     ),
#     tools_called=[
#         ToolCall(
#             name="Itinerary Generator",
#             description="Creates travel plans based on destination and duration.",
#             input_parameters={"destination": "Paris", "days": 3},
#             output=[
#                 "Day 1: Eiffel Tower, Le Jules Verne.",
#                 "Day 2: Louvre Museum, Angelina Paris.",
#                 "Day 3: Montmartre, wine bar.",
#             ],
#         ),
#         ToolCall(
#             name="Restaurant Finder",
#             description="Finds top restaurants in a city.",
#             input_parameters={"city": "Paris"},
#             output=["Le Jules Verne", "Angelina Paris", "local wine bars"],
#         ),
#     ],
# )

test_case = LLMTestCase(
    input=deepeval_trace['input'],
    actual_output=deepeval_trace['actual_output'],
    tools_called=deepeval_trace['tools_called'],
)

metric.measure(test_case)
print(metric.score)
print(metric.reason)

# or evaluate test cases in bulk
#evaluate([test_case], [metric])

In [ ]:
evaluate([test_case], [metric])